# 数据清洗与加工

In [ ]:
import json
import jieba
import jieba.analyse
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示符号
#修改相对路径位置
os.chdir(r'C:\Users\李世昌\PycharmProjects\PythonProject')

读取数据

In [ ]:
with open('./main/resident_evil_requiem_reviews.json', 'r', encoding='utf-8') as f:
    reviews=json.load(f)
df=pd.DataFrame(reviews)
df

清洗评论文本

In [ ]:
def clean_text(text):
    # 清洗，移除特殊字符（使用Python re模块支持的语法）
    text = re.sub(r'[\s\W_]+', ' ', text)
    # 清洗，移除多余空格
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df['clean_content']=df['content'].apply(clean_text)
#去重,去除空值
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
df

提取关键词

In [ ]:
def extract_keywords(text):
                            #text为清洗后的文本，topK为提取关键词的数量
    keywords = jieba.analyse.extract_tags(text, topK=10, withWeight=False)
    return keywords
df['keywords']=df['clean_content'].apply(extract_keywords)
df

分类反馈，基于分类关键词对清洗后的评论文本进行分类

In [ ]:
from collections import defaultdict
def classify(text):
    #定义分类关键词,关键词越多,结果越准确
    categories = {
        '画面': ['画面', '画质', '视觉', '特效', '场景', '分辨率','高清','超清','模糊','光影'],
        '剧情': ['剧情', '故事', '情节', '叙事', '结局', '紧凑', '剧情精彩','节奏','主线','世界观','人设','对话'],
        '玩法': ['玩法', '操作', '系统', '机制', '玩法创新', '操作流畅','手感','自由度','战斗'],
        '音效': ['音效', '音乐', '配音', '声音', 'bgm', 'BGM','音质','语音'],
        '性能': ['性能', '优化', '卡顿', '流畅度', '帧率', '丝滑','fps','FPS','闪退','掉帧','发热','配置','卡死'],
        '价格': ['价格', '性价比', '贵', '便宜', '性价比高', '价格合理','售价','打折','内购','氪金'],
        'bug': ['bug', '错误', '问题', '崩溃', '漏洞', '卡退','报错','穿模','补丁','修复'],
        '推荐': ['推荐', '值得', '必玩', '好评', '强烈推荐', '值得购买','爽','牛','强推']
    }
    d=defaultdict(int)
    max=0
    class_='其他'
    for category in categories:
        for keyword in categories[category]:
            if keyword in text:
                d[category]+=1
        if d[category]>max:
            max=d[category]
            class_=category
    return class_   #文本中关于哪一类的关键词个数最多,就识别为哪一类评论,初始类别默认为‘其他’
df['category']=df['clean_content'].apply(classify)
df

分析情感倾向

In [ ]:
def sentiment(text):
    senti='中性' #设置初始默认情感倾向
    if text=='推荐':
        senti='正面'
    elif text=='不推荐':
        senti='负面'
    return senti
df['sentiment']=df['recommendation'].apply(sentiment)
df

保存清洗加工后的表

In [ ]:
df.to_csv('./main/processed_table.csv')

# 数据可视化

情感分布

In [ ]:
plt.figure(figsize=(10,6),dpi=80)
sentiment_distribute=df['recommendation'].value_counts()
sns.barplot(x=sentiment_distribute.index,y=sentiment_distribute.values)
plt.title('玩家情感分布',fontsize=27)
plt.xlabel('情感',fontsize=12)
plt.ylabel('数量',fontsize=12)
plt.savefig('./main/情感分布.png')
plt.show()

游戏时长分布

In [ ]:
plt.figure(figsize=(10,6),dpi=80)
plt.hist(df['hours'],bins=20)
plt.title('游戏时长分布')
plt.xlabel('时长')
plt.ylabel('人数')
plt.savefig('./main/游戏时长分布.png')
plt.show()

玩家等级分布

In [ ]:
plt.figure(figsize=(10,6),dpi=80)
plt.hist(df['player_level'],bins=20)
plt.title('玩家等级分布',fontsize=27)
plt.xlabel('玩家等级',fontsize=12)
plt.ylabel('人数',fontsize=12)
plt.savefig('./main/玩家等级分布.png')
plt.show()

拥有游戏数量分布

In [ ]:
plt.figure(figsize=(10,6),dpi=80)
plt.hist(df['owned_games'],bins=20)
plt.title('拥有游戏数量分布',fontsize=27)
plt.xlabel('游戏数量',fontsize=12)
plt.ylabel('人数',fontsize=12)
plt.savefig('./main/拥有游戏数量分布.png')
plt.show()

评论类别分布

In [ ]:
plt.figure(figsize=(12,6),dpi=80)
class_distribute=df['category'].value_counts()
sns.barplot(x=class_distribute.index,y=class_distribute.values)
plt.title('评论类别分布',fontsize=27)
plt.xlabel('类别',fontsize=12)
plt.ylabel('评论数量',fontsize=12)
plt.savefig('./main/评论类别分布.png')
plt.show()

情感--分类交叉分析

In [ ]:
plt.figure(figsize=(12,6),dpi=160)
cross=pd.crosstab(df['category'],df['sentiment'])
cross.plot(kind='bar',stacked=True)
plt.title('情感--分类交叉分析',fontsize=27)
plt.xlabel('分类',fontsize=12)
plt.ylabel('情感数量',fontsize=12)
plt.legend(title='情感')
plt.xticks(rotation=27)
plt.savefig('./main/情感_分类交叉分析.png')
plt.show()

# 统计分析

In [ ]:
total=df['sentiment'].count() #共计3513条评论
positive=(df['sentiment']=='正面').sum()
negative=(df['sentiment']=='负面').sum()
neutral=(df['sentiment']=='中性').sum()

print(f'总评论数:{total}')
print(f'正面评论数:{positive} 占比{(positive/total)*100:.1f}%')
print(f'负面评论数:{negative} 占比{(negative/total)*100:.1f}%')
print(f'中性评论数:{neutral} 占比0')

In [ ]:
#分类统计
cate=df['category'].value_counts() #cate为series对象
for a in cate.index: #a:类别 b:该类别数量
    b=cate[a]
    print(f'{a}:{b} 占比{(b/total)*100:.1f}%')